# Foundry IQ: Topic Graph を使った Agentic Retrieval

## サマリー

**複数の文書にまたがる「テーマの質問」を Foundry IQ のナレッジ ベースに投げ、Topic Graph が根拠の選び方をどう変えるかを、実際の `retrieve` の応答で確かめます。**

「文書全体には、歴史、建造物、年中行事、観光についてどのような主要テーマがありますか」のような質問は、答えが 1 つの段落に収まりません。質問に最も近いチャンクだけを返す検索では、似た内容に偏ることがあります。Topic Graph は、複数のテーマから根拠を選ぶことで、幅広いテーマを反映した根拠を得やすくする機能です。

| 部品 | 役割 | このノートブックでの使い方 |
|---|---|---|
| Foundry IQ（Azure AI Search） | エージェントやアプリが、散在する文書から権限を考慮した根拠付きの情報を取り出すためのサービスです。検索対象を「ナレッジ ソース」、検索の窓口を「ナレッジ ベース」として定義します | 専用のナレッジ ソースとナレッジ ベースを 1 つずつ作成し、`retrieve` を呼び出します |
| Agentic Retrieval | ナレッジ ベースに `retrieve` を要求すると動く、複雑な質問向けのマルチクエリのパイプラインです。必要に応じて LLM が質問を分解し、並列に検索して再ランクし、結果を 1 つにまとめます | 多様性の設定だけを変えて同一条件で比較するため、`minimal` に固定して呼び出します。LLM による質問の分解、再ランク、回答の生成は使いません |
| Topic Graph | 関連するチャンクのまとまり（コミュニティ）と、そのトピック階層を表すグラフです。複数のトピックから根拠をサンプリングします | チャンクのベクトル フィールドに接続し、根拠の選び方を `none`・`balanced`・`diverse` で切り替えて比較します |

### 進め方

1. **1 章:** 接続設定と、Azure への変更を許可するスイッチを確認します。
2. **2.1〜2.2:** 登録済みのチャンク用インデックスとトピック用インデックスを確認し、チャンクのベクトル フィールドに Topic Graph を接続します。接続の追加は、計画を確認して承認した場合にだけ行います。
3. **2.3:** ナレッジ ソースとナレッジ ベースを作成します。
4. **2.4〜2.8:** 同じ質問を 3 方式で検索し、返ったチャンク、出典の参照、実行履歴を並べて比較します。

### 前提と注意

- **前提:** [機能テスト 2](./functional-test-2-topic-graph.ipynb) で構築したチャンク用インデックス（既定名 `ft-topic-src`）と、同じ機能テスト 2 の 9 章で Search に登録したトピック用インデックスです。このノートブックはグラフの構築や再登録を行いません。自分のデータで試す場合は、そのチャンク用インデックスに対応するトピック用インデックスを Foundry IQ skills プラグインで用意します。
- **プレビュー:** Topic Graph はパブリック プレビューの機能で、対象の API は `2026-11-01-preview` です。
- **結果の読み方:** 文書が少ないと、3 方式の結果が同じになることがあります。この比較は設定と返却データの確認であり、検索品質や多様性の改善を保証する評価ではありません。

出典: [What is Foundry IQ?](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq) / [Agentic retrieval in Foundry IQ](https://learn.microsoft.com/azure/search/agentic-retrieval-overview) / [Topic Graph の検索構成](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Knowledge graph features overview](https://learn.microsoft.com/azure/search/knowledge-graph-features-overview)

## 背景: Foundry IQ、Agentic Retrieval、Topic Graph

### Foundry IQ は何を提供するか

エージェントを動かすモデルには学習データの時点という知識の限界があり、組織内の文書には自力でアクセスできません。Foundry IQ は、複数のデータ ストアにある文書を 1 つの「ナレッジ ベース」として束ね、エージェントやアプリに、権限を考慮した根拠付きの情報を返します。1 つのナレッジ ベースを複数のエージェントで共有できます。

図書館にたとえると、**ナレッジ ソースは「書庫」、ナレッジ ベースは「司書の窓口」** です。利用者は窓口に質問を出すだけで、どの書庫をどう探すかは窓口が決めます。書庫には Azure Blob Storage、SharePoint、OneLake、公開 Web などを指定できます。このノートブックの書庫は、Search サービス上のチャンク用インデックス 1 つです。

```mermaid
flowchart LR
    U["エージェント / アプリ"] -->|"retrieve（質問）"| KB["ナレッジ ベース<br/>司書の窓口"]
    KB --> KS1["ナレッジ ソース<br/>このノートブック: チャンク用インデックス"]
    KB -.-> KS2["他のナレッジ ソース<br/>Blob、SharePoint、OneLake、Web"]
    KS1 --> R["応答<br/>根拠の抜粋・出典の参照・実行履歴"]
    KS2 -.-> R
    R --> U
```

### Agentic Retrieval は何をするか

Agentic Retrieval は、チャットやエージェントからの複雑な質問を想定した、マルチクエリのパイプラインです。たとえば「ビーチの近くで、空港の送迎があり、ベジタリアン向けのレストランまで歩けるホテル」のように複数の依頼を含む質問は、1 回の検索では扱いにくいため、質問を分けて並列に検索します。応答は、根拠の抜粋（`response`）、出典の参照（`references`）、実行履歴（`activity`）で構成されます。

| 段階 | 通常の動き | このノートブックの設定 |
|---|---|---|
| 1. 計画 | `low`・`medium` では、LLM が質問をサブクエリに分解し、検索するナレッジ ソースを選びます | `minimal`: LLM を使わず、質問を 1 つの検索意図としてそのまま渡します |
| 2. 実行 | サブクエリを並列に実行し、各結果をセマンティック ランカーで再ランクします | ナレッジ ソースの `resultsProcessing = none`: 再ランクを省き、インデックスの結果の順序を保ちます |
| 3. 統合 | 抜粋を返すか、LLM が回答を合成します | `extractiveData`: 抜粋と出典の参照を返し、回答は生成しません |

**推論レベルを `minimal` に固定する理由は、多様性の設定を同一条件で比較するためです。** 2.5〜2.7 は、同じ質問、同じナレッジ ソースに対して `topicGraph` の設定（`none`・`balanced`・`diverse`）だけを変えます。`low`・`medium` では LLM が質問をサブクエリに分解するため、要求ごとに実際に実行される検索が変わり得て、結果の違いを多様性の設定の違いとして読めなくなります。`minimal` は LLM による検索計画を省き、質問を 1 つの検索意図としてそのまま渡します。再ランクを省く `resultsProcessing = none` も同じ観点の設定で、返却順に再ランクの影響が入りません。

このノートブックは、Agentic Retrieval の枠組み（ナレッジ ベース、ナレッジ ソース、`retrieve`）を使いながら、LLM を使う部分を切り離し、Topic Graph による根拠の選び方の違いだけを観察する構成です。LLM による検索計画を使う例は、[Entity Graph のノートブック](./agentic-retrieval-entity-graph.ipynb) で扱います。

### Topic Graph は何を解決するか

ベクトル検索は、質問に最も近いチャンクを上位から返します。「この文書群にはどのようなテーマがありますか」のように全体を問う質問では、似た内容のチャンクが上位を占めると、ほかのテーマが根拠から漏れます。

Topic Graph は、関連する内容のまとまり（コミュニティ）と、それらが作るトピックの階層を表します。複数のトピックから根拠をサンプリングすることで、最も近いベクトルだけを返す場合より、幅広いテーマを反映した根拠を得やすくなります。図書館にたとえると、「この図書館にはどのような分野の本がありますか」と聞かれたときに、質問の言葉に最も近い 1 冊の周辺だけを集めず、複数の分野の棚から根拠を集めるイメージです。棚ごとに何冊ずつ選ぶか、階層をどの順に調べるかといった内部処理は公開されていません。公開資料で確認できる処理順序は、2.3 の後ろにある「`retrieve` の処理順序」にまとめています。

同じグラフ機能でも、Entity Graph は人、組織、場所などの実体どうしのつながりをたどります（例: どの施設がどの指定を受けているか）。Topic Graph が補助するのはテーマを横断する検索で、オントロジーは不要です。このノートブックの `retrieve` は GQL を生成せず、実行履歴に `graphSearch` が出ないのは正常です（2.8.1）。

コミュニティで文書群全体のテーマを捉える考え方は、GraphRAG の研究でも扱われています。ただし、Foundry IQ の内部処理が同じかどうかは公開されていません。

出典: [What is Foundry IQ?](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq) / [Agentic retrieval の構成と流れ](https://learn.microsoft.com/azure/search/agentic-retrieval-overview#architecture-and-workflow) / [`resultsProcessing` による再ランクの省略](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#call-the-retrieve-action) / [Topic Graph の概要](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Topic Graph と Entity Graph の比較](https://learn.microsoft.com/azure/search/knowledge-graph-features-overview#compare-topic-graphs-and-entity-graphs) / Edge et al., [From Local to Global: A Graph RAG Approach to Query-Focused Summarization](https://arxiv.org/abs/2404.16130), 2024

### チャンク用インデックスから Topic Graph を参照する仕組み

ナレッジ ソースが検索するのはチャンク用インデックスです。Topic Graph を使う場合は、その検索対象のベクトル フィールドから、同じ Search サービスにあるトピック用インデックスを間接的に参照します。図の矢印は検索処理の順序ではなく、設定の参照関係を表します。

```mermaid
flowchart TD
    KB["ナレッジ ベース"] -->|"knowledgeSources"| KS["ナレッジ ソース（searchIndex）"]
    KS -->|"searchIndexParameters.searchIndexName"| C["チャンク用インデックス"]
    C --> V["検索対象のベクトル フィールド"]
    V -->|"topicGraphProfile"| P["同じインデックスのトピック プロファイル"]
    P -->|"indexName"| T["トピック用インデックス<br/>graphProfile: グラフ構造の定義"]
```

| 設定 | 役割 |
|---|---|
| `graphProfile` | トピック用インデックスでグラフの構造を定義します。チャンク用インデックスからの参照先を設定する項目ではありません。 |
| `topicGraphProfile` | 検索対象のベクトル フィールドに設定するプロファイル名です。チャンク用インデックスの `vectorSearch.topicGraphProfiles` から同名の項目を探し、その `indexName` でトピック用インデックスを参照します。 |
| `vectorSearchProfile` | 近傍検索とクエリのベクトル化に使うプロファイルです。質問はビルド時と同じ埋め込みモデルのベクトライザーで変換します。トピック用インデックスへの参照は別の `topicGraphProfile` が受け持ちます。 |

トピック サンプリングを有効にすると、Agentic Retrieval は**トピック用インデックス**を候補の選択に使います。根拠として返す本文は**チャンク用インデックス**から取得します。トピック用インデックスをナレッジ ソースに直接指定する必要はありません。2 つのインデックスが何でつながるかは、2.3 の後の「2 つのインデックスをつなぐもの」で確認します。

この Notebook ではナレッジ ソースの既定を `balanced`（多様性係数 `0.2` 固定）に設定します。2.5 の `none` はトピック サンプリングを無効にする基準、2.7 の `diverse` は `diversityFactor` を `0`〜`1` で要求ごとに変える設定です。係数を変えても上位の出典が変わるとは限りません。違いは 2.8 で実際の検索応答を比較して確認します。

**自分の環境で確認するには:** `CONTENT_INDEX` にチャンク用インデックス、`TOPIC_INDEX` にそのチャンクに対応する登録済みのトピック用インデックス、`VECTOR_FIELD` に検索対象のベクトル フィールドを指定し、2.2 で参照関係を確認します。既存のナレッジ ベースとナレッジ ソースの設定が一致していれば、作り直す必要はありません。

出典: [Microsoft Learn: Topic Graph の参照先を確認する](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#set-up-requests) / [リクエスト単位でトピック選択を上書きする](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#override-topic-selection-for-one-request) / [設定値と既定値](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#review-settings-and-verification-limits)

## 1. 接続設定と変更の境界

### Azure 側の準備

| 項目 | 必要なもの |
|---|---|
| サービス | `2026-11-01-preview` とグラフ機能に対応する Foundry IQ / Azure AI Search |
| Topic Graph | 機能テスト 2 で構築した `ft-topic-src` と、機能テスト 2 の 9 章で登録したトピック インデックス。ビルド時と同じ埋め込みモデル・デプロイを使います |
| 実行ユーザー | 参照には Search Index Data Reader。新規構成やインデックス設定変更には Search Service Contributor も必要です。既存の許可で足りれば追加付与は不要です |
| Search のマネージド ID | Azure OpenAI への Cognitive Services OpenAI User。ユーザーがサインインできても、サービスからモデルへ接続できるとは限りません |

この Notebook はサービスやモデルの作成、ロール付与、文書のアップロード、グラフの再抽出、リソース削除を行いません。

### カーネルの依存パッケージ

ライブ用セルで import エラーが出る場合、または SDK が `12.2.0b1` でない場合は、選択した Notebook カーネルで次を実行してください。機能テスト 2 と同じバージョンに固定します。

```python
%pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv
```

[.env.example](./.env.example) を参考に、このノートブックと同じフォルダーに `.env` を用意し、Azure CLI などで対象テナントへサインインします。資格情報やアクセストークンをセルへ直接書かないでください。機能テスト 2 と同じく `.env` の設定を優先します。

### 環境変数と既定値

このノートブックはチャット モデルを使わないため、`AZURE_OPENAI_DEPLOYMENT` は不要です。

| 環境変数 | 必要になる場面 | 未設定のときの既定 |
|---|---|---|
| `AZURE_SEARCH_ENDPOINT` | `RUN_AZURE = True` のとき必須です | なし |
| `AZURE_OPENAI_ENDPOINT` | 2.2 で、検索フィールドにベクトライザーがなく追加が必要な場合 | なし |
| `AZURE_OPENAI_EMBEDDING_DEPLOYMENT` | 同上 | なし |
| `AZURE_OPENAI_EMBEDDING_MODEL` | 同上。デプロイ名とモデル名が異なる場合に設定します | 埋め込みデプロイ名（機能テスト 2 の `AZURE_OPENAI_MODEL` が未設定ならデプロイ名を使うのと同じ扱い） |
| `FOUNDRY_IQ_TOPIC_INDEX` | 登録済みのトピック インデックスを明示する場合 | 空。機能テスト 2 の 9.3 で保存した記録、または `ft-published-topics-` で始まるインデックスから探します（2.1） |

このノートブックが使うインデックスとナレッジ ソース、ナレッジ ベースの名前は、機能テスト 2 と同じ接頭辞 `FT = "ft-topic"` から決まります。別の名前を使う場合は、設定セルの `FT` を変更します。

| 対象 | 既定名 |
|---|---|
| チャンク用インデックス | `ft-topic-src` |
| ナレッジ ソース | `ft-topic-agentic-ks` |
| ナレッジ ベース | `ft-topic-agentic-kb` |

### 実行スイッチ

- `RUN_AZURE`: 既定は `True` で、Azure への接続と検索を行います。接続しない場合は `False` にします。質問の埋め込みには課金が発生する場合があります。
- `ENABLE_TOPIC`: 既定は `True` です。`False` にすると Topic Graph の検索と設定変更をスキップします。
- `CONFIRM_WRITES`: 既定は `False` です。計画を確認し、変更が必要な場合だけ `True` にして設定セルを再実行します。この設定は、専用ナレッジ ベース・ソースの新規作成と、検証用 `ft-topic-src` の検索設定変更を承認します。既存ソースや既存ナレッジ ベースを上書きしません。

出典: [Topic Graph の前提](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#prerequisites) / [ナレッジ ベースの検索権限](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#prerequisites) / [Search の RBAC](https://learn.microsoft.com/azure/search/search-security-rbac) / [ローカルでの Azure Identity 認証](https://learn.microsoft.com/azure/developer/python/sdk/authentication/local-development-environment)

In [ ]:
from __future__ import annotations

import copy
import html
import json
import os
from pathlib import Path
from urllib.parse import urlsplit

from IPython.display import HTML, SVG, display


def show_json(value: object) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def preview_text(value: object) -> object:
    return value[:200] + "…" if isinstance(value, str) and len(value) > 200 else value


def table(rows: list[dict]) -> None:
    if not rows:
        print("表示対象は 0 件です。")
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    header = "".join(f"<th>{html.escape(str(key))}</th>" for key in columns)
    body = []
    for row in rows:
        cells = []
        for key in columns:
            value = row.get(key, "")
            text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
            cells.append(f"<td>{html.escape(text)}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"))

### `CONTENT_INDEX` と `TOPIC_INDEX` に何を指定するか

**根拠本文を取り出す先が `CONTENT_INDEX`、根拠候補を話題ごとに選ぶための参照先が `TOPIC_INDEX` です。** 社内マニュアルでたとえると、前者には申請手順などの本文が入り、後者には「申請」「承認」などの話題と、それぞれに属する本文の断片の対応が入ります。

| 設定 | コピーする名前 | このリポジトリでの命名規則 |
|---|---|---|
| `CONTENT_INDEX` | Topic Graph の構築時に入力として使ったチャンク用インデックス名です。[機能テスト 2 の 1.2 節と 4 章](./functional-test-2-topic-graph.ipynb) では `SRC_INDEX` の値を `search.sourceIndex` に渡しています。 | `<FT>-src`。`FT` はテスト用の接頭辞で、機能テスト 2 の既定値は `ft-topic` です。 |
| `TOPIC_INDEX` | そのチャンクから作った Topic Graph を Azure AI Search に登録した際のインデックス名です。[機能テスト 2 の 9.3 節](./functional-test-2-topic-graph.ipynb) の登録結果、または `.functional-test/published-resources.json` の `topic_index` からコピーします。 | `<PREFIX>-topics-<ハッシュの先頭12桁>`。`PREFIX` の既定値は `ft-published` です。ハッシュは Topic Graph のスキーマと登録文書から計算します。 |

指定するのは、**同じ Search サービスにある、同じビルドの入力と登録先に対応する別々のインデックス名**です。命名規則は機能テストでの目印です。名前を組み立て直さず、実際に構築・登録したときの値を使います。機能テスト 2 のローカル出力は `TOPIC_INDEX` には指定できません。未登録なら先に Topic Graph を Search に登録します。

すでに接続済みなら、ナレッジ ソースの `searchIndexParameters.searchIndexName` が `CONTENT_INDEX` です。チャンク用インデックスで検索対象のベクトル フィールドが指定する `topicGraphProfile` をたどり、同名のプロファイルの `indexName` を確認すると `TOPIC_INDEX` が分かります。[Microsoft Learn: Topic Graph の参照設定](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#set-up-requests)

In [ ]:
RUN_AZURE = True
ENABLE_TOPIC = True
CONFIRM_WRITES = False
API_VERSION = "2026-11-01-preview"

FT = "ft-topic"  # 機能テスト 2 と同じ接頭辞
NOTEBOOK_DIR = Path.cwd()

if RUN_AZURE:
    from importlib.metadata import version
    from azure.core import MatchConditions
    from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
    from azure.core.pipeline import PipelineResponse
    from azure.identity import DefaultAzureCredential
    from azure.search.documents import SearchClient
    from azure.search.documents.indexes import SearchIndexClient
    from azure.search.documents.indexes.models import (
        KnowledgeBase, KnowledgeSource, KnowledgeSourceReference, SearchIndex, SearchIndexFieldReference,
        SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters,
    )
    from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
    from azure.search.documents.knowledgebases.models import (
        KnowledgeBaseRetrievalRequest, KnowledgeRetrievalMinimalReasoningEffort,
        KnowledgeRetrievalSemanticIntent, SearchIndexKnowledgeSourceParams,
    )
    from azure.search.documents.models import (
        BalancedVectorQueryTopicGraph, DiverseVectorQueryTopicGraph, NoneVectorQueryTopicGraph,
    )
    from dotenv import load_dotenv
    if version("azure-search-documents") != "12.2.0b1":
        raise RuntimeError("機能テスト 2 と同じ azure-search-documents==12.2.0b1 をカーネルに導入してください。")
    load_dotenv(NOTEBOOK_DIR / ".env", override=True)
    credential = DefaultAzureCredential()

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "").rstrip("/")
MODEL_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
EMBED_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "")
EMBED_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", EMBED_DEPLOYMENT)
CONTENT_INDEX = f"{FT}-src"
VECTOR_FIELD = "snippet_vector"
CONTENT_FIELD = "snippet"
TOPIC_INDEX = os.getenv("FOUNDRY_IQ_TOPIC_INDEX", "")
TOPIC_SOURCE = f"{FT}-agentic-ks"
TOPIC_KB = f"{FT}-agentic-kb"

if RUN_AZURE:
    endpoint = urlsplit(SEARCH_ENDPOINT)
    if (endpoint.scheme != "https" or not endpoint.hostname or endpoint.username
            or endpoint.password or endpoint.path or endpoint.query or endpoint.fragment or "<" in SEARCH_ENDPOINT):
        raise ValueError("AZURE_SEARCH_ENDPOINT に実際の HTTPS サービス URL を設定してください。")
    client_options = {"api_version": API_VERSION, "retry_total": 0, "connection_timeout": 10, "read_timeout": 90}
    index_client = SearchIndexClient(SEARCH_ENDPOINT, credential, **client_options)
    azure_clients: list[SearchIndexClient | SearchClient | KnowledgeBaseRetrievalClient] = [index_client]
    if ENABLE_TOPIC and not TOPIC_INDEX:
        state_path = Path(".functional-test/published-resources.json")
        if state_path.is_file():
            state = json.loads(state_path.read_text(encoding="utf-8"))
            if state["endpoint"].rstrip("/") != SEARCH_ENDPOINT or state["api_version"] != API_VERSION:
                raise ValueError("機能テスト 2 の保存先と接続先・API が異なります。登録先を確認してください。")
            TOPIC_INDEX = state.get("topic_index", "")
        if not TOPIC_INDEX:
            candidates = sorted(name for name in index_client.list_index_names() if name.startswith("ft-published-topics-"))
            table([{"Topic Graph 候補": name} for name in candidates])
            if len(candidates) != 1:
                raise ValueError("候補が 0 件または複数です。機能テスト 2 の 9 章の登録を確認し、FOUNDRY_IQ_TOPIC_INDEX に登録名を指定してください。")
            TOPIC_INDEX = candidates[0]

table([{"Azure 実行": RUN_AZURE, "Topic 実行": ENABLE_TOPIC, "書き込み承認": CONFIRM_WRITES,
        "Topic インデックス": TOPIC_INDEX or "未指定"}])

### 1.1 SDK の例外処理と安全な新規作成

認証と通信は SDK に任せます。サービスのエラーは SDK の `HttpResponseError` で停止し、存在しないリソースだけ `ResourceNotFoundError` として扱います。`retrieve` の部分応答 `206` は応答フックで検出して停止します。各クライアントの `retry_total = 0` により、自動再試行は行いません。

新規リソースは `MatchConditions.IfMissing` で既存の上書きを防止します。再実行時に存在する場合は、この Notebook の計画に含まれる設定値を照合してから再利用します。サーバーが追加する既定値は照合対象外です。

出典: [Search の同時実行制御](https://learn.microsoft.com/azure/search/search-howto-concurrency) / [Knowledge Retrieval REST](https://learn.microsoft.com/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-11-01-preview)

In [ ]:
def reject_partial_response(response: PipelineResponse) -> None:
    if response.http_response.status_code == 206:
        raise HttpResponseError("Agentic Retrieval が部分応答を返しました。応答本文と実行履歴を確認してください。",
                                response=response.http_response)


def require_expected(actual: object, expected: object, path: str = "resource") -> None:
    if isinstance(expected, dict) and isinstance(actual, dict):
        for key, value in expected.items():
            require_expected(actual.get(key), value, f"{path}.{key}")
    elif isinstance(expected, list) and isinstance(actual, list) and len(actual) == len(expected):
        for index, value in enumerate(expected):
            require_expected(actual[index], value, f"{path}[{index}]")
    elif actual != expected:
        raise ValueError(f"既存設定が計画と異なります: {path}。上書きせず、別名を使うか設定を確認してください。")


def create_or_reuse(definition: KnowledgeBase | SearchIndexKnowledgeSource) -> KnowledgeBase | KnowledgeSource:
    if not RUN_AZURE:
        raise RuntimeError("Azure 呼び出しは無効です。RUN_AZURE を確認してください。")
    try:
        if isinstance(definition, KnowledgeBase):
            existing = index_client.get_knowledge_base(definition.name)
        else:
            existing = index_client.get_knowledge_source(definition.name)
    except ResourceNotFoundError:
        print("未作成:", definition.name)
        if not CONFIRM_WRITES:
            raise RuntimeError("計画を確認し、CONFIRM_WRITES = True にしてこのセルから再実行してください。") from None
        if isinstance(definition, KnowledgeBase):
            return index_client.create_or_update_knowledge_base(definition, match_condition=MatchConditions.IfMissing)
        return index_client.create_or_update_knowledge_source(definition, match_condition=MatchConditions.IfMissing)
    require_expected(existing.as_dict(), definition.as_dict())
    print("設定を照合して再利用:", definition.name)
    return existing

## 2. Topic Graph で文書群のテーマを検索する

### 2.1 コンテンツとトピックの定義・登録データを確認する

`FOUNDRY_IQ_TOPIC_INDEX` が空なら、機能テスト 2 の 9.3 が保存した `.functional-test/published-resources.json` から登録先を読みます。記録がない場合は、機能テスト 2 の 9.1 と同じく `ft-published-topics-` で始まるインデックスを SDK で探します。候補が 1 件なら再利用し、0 件または複数なら停止します。別の登録先は接続設定で明示してください。

両インデックスにデータがあることを確認します。埋め込みの次元も確認しますが、**同じ次元だけでは、同じモデルで生成されたベクトルだとは証明できません。** ビルド時の埋め込みデプロイとモデルを設定してください。

出典: [Topic Graph の前提条件と定義確認](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#prerequisites)

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    content_schema = index_client.get_index(CONTENT_INDEX).as_dict()
    topic_schema = index_client.get_index(TOPIC_INDEX).as_dict()
    content_vector = next(field for field in content_schema["fields"] if field["name"] == VECTOR_FIELD)
    topic_dimensions = {field["dimensions"] for field in topic_schema["fields"] if field.get("dimensions")}
    if content_vector["dimensions"] not in topic_dimensions:
        raise ValueError("トピックとコンテンツのベクトル次元が一致しません。ビルド時の設定を確認してください。")
    for schema in (content_schema, topic_schema):
        key_name = next(field["name"] for field in schema["fields"] if field.get("key"))
        with SearchClient(SEARCH_ENDPOINT, schema["name"], credential, **client_options) as source_client:
            sample = source_client.search("*", select=[key_name], top=1, include_total_count=True)
            count = sample.get_count()
            if count is None:
                raise RuntimeError("登録件数を取得できません。include_total_count の応答を確認してください。")
            table([{"インデックス": schema["name"], "登録件数": count, "キーの例": list(sample)}])
        if count == 0:
            raise ValueError("インデックスが空です。機能テスト 2 のビルドと 9 章の登録を完了してください。")
    show_json(content_vector)
else:
    print("未実行: Topic Graph の定義とデータの確認。RUN_AZURE と ENABLE_TOPIC が必要です。")

### 2.2 検索フィールドに Topic Graph を接続する計画

**この節の目的:** 2.5〜2.7 の Topic Graph 検索（`none`・`balanced`・`diverse`）は、検索するチャンク用インデックスのベクトル フィールドが、トピック用インデックスに接続されていることを前提にします。公式の前提条件も、検索用のベクトル フィールドを用意済みのトピック用インデックスにリンクしておくことを求めており、リンクが欠けている場合は、検索対象フィールドの `topicGraphProfile` を確認するよう案内しています。質問文をベクトルに変換するため、そのフィールドにはベクトライザーも必要です。

**公式手順との関係:** Microsoft Learn の手順は、接続済みのインデックスを前提にしており、接続を後から追加する手順は載っていません。さらに、その手順の一部として置き換え用のインデックス定義を送らないことを求めています。接続が無い場合の設定は、Foundry IQ skills プラグインの TopicGraph セットアップ（`plan-retrieval` → `configure-retrieval` → `verify-retrieval`）が担当します。プラグインの資料によると、検索対象のベクトル フィールドへのトピック接続の設定と、既存のナレッジ ソースの調整を、承認付きで行います。この節は、同じ種類の設定を SDK で行うこの Notebook の実装で、プラグインの処理そのものではありません。接続の構造（フィールドの `topicGraphProfile` と `vectorSearch.topicGraphProfiles`）は公式の REST 定義例に従い、接続とベクトライザーが揃っていれば何も変更しません。

機能テスト 2 の 9 章はトピックを登録しますが、検索フィールドの `topicGraphProfile` と質問を埋め込むベクトライザーは設定しません。ここで検証用 `ft-topic-src` の既存定義をコピーし、必要な検索設定だけを追加します。**既存フィールドの `vectorSearchProfile` 参照と検索アルゴリズムは変更しません。**

接続が未設定の場合はトピック プロファイルを追加します。ベクトライザーがなければ、同じ埋め込みモデルのベクトライザーを追加し、既存プロファイルへその参照を設定します。既存のベクトライザーがある場合は再利用します。既に別の Topic Graph が接続されている場合は停止します。この自動設定は `ft-topic-src` のみに限定します。名前を変更して運用インデックスへ適用しないでください。

定義取得時の ETag を更新用 `SearchIndex` の `e_tag` に設定し、`MatchConditions.IfNotModified` で更新します。他の変更が入った場合は `412` で停止します。更新前の定義はカーネル内の `content_schema` に保持しますが、認証情報が含まれる可能性を考慮し、ファイル保存はしません。

出典: [Topic Graph の前提条件](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#prerequisites) / [Topic Graph のトラブルシューティング](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#troubleshooting) / [Topic Graph の接続確認](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#set-up-requests) / [Foundry IQ skills: 検索設定の手順（retrieval-setup）](https://github.com/microsoft/azure-skills/blob/7847020998043c6231929b364c35393b7cdab270/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/retrieval-setup.md) / [Foundry IQ skills: トピック多様性の前提設定](https://github.com/microsoft/azure-skills/blob/7847020998043c6231929b364c35393b7cdab270/.github/plugins/foundry-iq-skills/graph/apps-host/standalone-topic-diversity.md#native-setup-prerequisite) / [トピック接続の公式 REST 定義例](https://github.com/Azure/azure-rest-api-specs/blob/be93aca8c35ada896dd7fd6c7f65b1a9f766e84c/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchServiceCreateOrUpdateIndexWithTopicGraphs.json) / [既存インデックスへのベクトライザー追加](https://learn.microsoft.com/azure/search/vector-search-how-to-configure-vectorizer#define-a-vectorizer-programmatically) / [ベクトル化済みフィールドの更新制限](https://learn.microsoft.com/azure/search/vector-search-how-to-create-index#update-a-vector-index) / [同時実行制御](https://learn.microsoft.com/azure/search/search-howto-concurrency)

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    topic_index_plan = None
    vector_search = content_schema["vectorSearch"]
    old_profile = next(profile for profile in vector_search["profiles"] if profile["name"] == content_vector["vectorSearchProfile"])
    bound_name = content_vector.get("topicGraphProfile")
    bound_profile = next((profile for profile in vector_search.get("topicGraphProfiles", []) if profile["name"] == bound_name), None)
    existing_vectorizer = next((item for item in vector_search.get("vectorizers", []) if item["name"] == old_profile.get("vectorizer")), None)
    if bound_name and (bound_profile is None or bound_profile["indexName"] != TOPIC_INDEX):
        raise ValueError("検索フィールドに別のトピック接続があります。上書きせず設定を確認してください。")
    if bound_profile is not None and existing_vectorizer is not None:
        print("Topic Graph とベクトライザーは既に接続済みです。更新しません。")
    else:

        topic_index_plan = {key: copy.deepcopy(value) for key, value in content_schema.items() if not key.startswith("@odata.")}
        planned_search = topic_index_plan["vectorSearch"]
        planned_profile = next(profile for profile in planned_search["profiles"] if profile["name"] == old_profile["name"])
        additions = {}
        if bound_profile is None:
            additions["topicGraphProfiles"] = {"name": "iq-notebook-topics", "indexName": TOPIC_INDEX,
                                                "defaultKind": "balanced", "defaultMinDiversityPoolSize": 1000}
        if existing_vectorizer is None:
            if not all((MODEL_ENDPOINT, EMBED_DEPLOYMENT, EMBED_MODEL)) or any("<" in value for value in (MODEL_ENDPOINT, EMBED_DEPLOYMENT, EMBED_MODEL)):
                raise ValueError("機能テスト 2 と同じ埋め込みデプロイと実モデル名を設定してください。")
            other_fields = [field["name"] for field in content_schema["fields"]
                            if field.get("vectorSearchProfile") == old_profile["name"] and field["name"] != VECTOR_FIELD]
            if other_fields:
                raise ValueError(f"プロファイルは他のフィールドでも使用されています: {other_fields}。自動変更を停止します。")
            additions["vectorizers"] = {"name": "iq-notebook-vectorizer", "kind": "azureOpenAI", "azureOpenAIParameters": {
                "resourceUri": MODEL_ENDPOINT, "deploymentId": EMBED_DEPLOYMENT, "modelName": EMBED_MODEL}}
            planned_profile["vectorizer"] = "iq-notebook-vectorizer"
        for collection, addition in additions.items():
            items = planned_search.setdefault(collection, [])
            existing = next((item for item in items if item["name"] == addition["name"]), None)
            if existing is None:
                items.append(addition)
            else:
                require_expected(existing, addition)
        planned_field = next(field for field in topic_index_plan["fields"] if field["name"] == VECTOR_FIELD)
        planned_field["topicGraphProfile"] = bound_name or "iq-notebook-topics"
        show_json({"変更対象": CONTENT_INDEX, "変更フィールド": planned_field,
                   "既存プロファイルへの質問用設定": planned_profile, "追加する検索設定": additions})
else:
    print("未実行: Topic Graph 接続の変更計画。")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    if topic_index_plan is not None:
        if not CONFIRM_WRITES:
            raise RuntimeError("埋め込みモデル・接続先・変更計画を確認し、CONFIRM_WRITES = True にしてください。")
        topic_index_model = SearchIndex(topic_index_plan)
        topic_index_model.e_tag = content_schema["@odata.etag"]
        index_client.create_or_update_index(topic_index_model, match_condition=MatchConditions.IfNotModified)
    effective_schema = index_client.get_index(CONTENT_INDEX).as_dict()
    effective_field = next(field for field in effective_schema["fields"] if field["name"] == VECTOR_FIELD)
    effective_search = effective_schema["vectorSearch"]
    effective_topic = next(profile for profile in effective_search["topicGraphProfiles"] if profile["name"] == effective_field["topicGraphProfile"])
    effective_vector = next(profile for profile in effective_search["profiles"] if profile["name"] == effective_field["vectorSearchProfile"])
    if effective_topic["indexName"] != TOPIC_INDEX:
        raise ValueError("読み戻した Topic Graph の接続先が計画と異なります。")
    effective_vectorizer = next(item for item in effective_search["vectorizers"] if item["name"] == effective_vector["vectorizer"])
    show_json({"トピック接続": effective_topic, "ベクトライザー名": effective_vectorizer["name"]})
else:
    print("未実行: 検証用インデックスの設定更新と読み戻し。")

### 2.3 トピック検索用ソースとナレッジ ベースを作る

検索するフィールドは `snippet_vector`、原文として返すフィールドは `uid`・`snippet`・`blob_url` です。ソースの既定は `balanced` とし、後から要求単位で変更します。`minimal` を使うナレッジ ベースには検索計画用のチャットモデルを設定しません。

作成は既定名の専用ソースと KB のみで、既存の異なる設定は上書きしません。ソースの `sourceDataFields` に `uid` を含めるのは、検索ごとに変わり得る参照 ID ではなく、同じチャンク ID で結果を比較するためです。

出典: [Topic Graph のソース作成](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#create-the-knowledge-source) / [ナレッジ ベースへの接続](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#attach-the-source-to-a-knowledge-base)

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    topic_source_plan = SearchIndexKnowledgeSource(
        name=TOPIC_SOURCE, results_processing="none",
        description="agentic-retrieval-graphs notebook: topic retrieval",
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=CONTENT_INDEX, search_fields=[SearchIndexFieldReference(name=VECTOR_FIELD)],
            source_data_fields=[SearchIndexFieldReference(name=name) for name in ("uid", CONTENT_FIELD, "blob_url")],
            topic_graph=BalancedVectorQueryTopicGraph(min_diversity_pool_size=1000),
        ),
    )
    show_json(topic_source_plan.as_dict())
    topic_source_definition = create_or_reuse(topic_source_plan)
    require_expected(topic_source_definition.as_dict(), topic_source_plan.as_dict())
else:
    print("未実行: Topic Graph 検索用ソースの作成・照合。")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    topic_kb_plan = KnowledgeBase(
        name=TOPIC_KB, description="agentic-retrieval-graphs notebook: topic retrieval",
        knowledge_sources=[KnowledgeSourceReference(name=TOPIC_SOURCE)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(), output_mode="extractiveData",
    )
    show_json(topic_kb_plan.as_dict())
    topic_kb_definition = create_or_reuse(topic_kb_plan)
    require_expected(topic_kb_definition.as_dict(), topic_kb_plan.as_dict())
    topic_kb_client = KnowledgeBaseRetrievalClient(SEARCH_ENDPOINT, credential, knowledge_base_name=TOPIC_KB, **client_options)
    azure_clients.append(topic_kb_client)
else:
    print("未実行: Topic Graph 検索用ナレッジ ベースの作成・照合。")

### `retrieve` の処理順序：候補取得からトピック サンプリングまで

**要点は、先にベクトル検索で候補を広く集め、その後でトピックを使って根拠を選び直す二段構えであることです。** 公開仕様で確認できる順序は、ベクトル検索の候補取得がトピック サンプリングに先立つことです。以下は、このノートブックが使う `2026-11-01-preview` で、公式資料から確認できる流れです。図の青い実線の枠は公式資料に記載がある処理、橙色の破線の枠は存在は確認できるものの内部の動作が公開されていない処理です。

```mermaid
flowchart TD
    Q["① 呼び出し側<br/>retrieve に質問 1 件（intents）と topicGraph の指定を渡す"]
    P["② 計画（minimal）<br/>モデルによる計画は行わず、対象のナレッジ ソースをすべて検索"]
    V["③ 質問をベクトル化<br/>検索フィールドのベクトライザーを使用"]
    C["④ ベクトル検索<br/>balanced と diverse は、サンプリングの前に<br/>少なくとも minDiversityPoolSize 件の候補を取得（既定 1000）"]
    S["⑤ トピック サンプリング<br/>トピックの代表ベクトルを使い、複数のテーマから根拠を選ぶ<br/>balanced: 係数 0.2 固定 / diverse: 係数 0〜1 を指定"]
    N["⑤' none<br/>サンプリングを行わず、通常のベクトル検索の結果を使う"]
    R["⑥ 応答（extractiveData）<br/>抜粋 response / 出典 references / 実行履歴 activity"]
    T[("トピック用インデックス<br/>トピック ID と重心ベクトル")]
    K[("チャンク用インデックス ft-topic-src<br/>チャンク本文と snippet_vector")]
    Q --> P --> V --> C
    K -->|"検索対象"| C
    C -->|"balanced / diverse"| S --> R
    C -->|"none"| N --> R
    K -.->|"抜粋の本文を取得"| R
    T -.->|"topicGraphProfile の indexName で参照"| S
    classDef known fill:#e8f2ff,stroke:#3976b8,stroke-width:1.5px;
    classDef opaque fill:#fff7e6,stroke:#d97706,stroke-width:2px,stroke-dasharray:5 4;
    class Q,P,V,C,N,R,T,K known;
    class S opaque;
```

トピック用インデックスをどの段階で参照するかは公開されていません。トピックの代表ベクトルを使うと説明されているサンプリングに結び付けて、点線で示しています。

#### 関係するインデックスは 2 つ

| インデックス | このノートブックでの名前 | 中身 | 使われ方 |
|---|---|---|---|
| チャンク用インデックス | `ft-topic-src`（`CONTENT_INDEX`） | 文書を分割したチャンクの本文とベクトル（`snippet_vector`） | ナレッジ ソースが直接指定する検索対象です。④ で検索し、⑥ で返す抜粋の本文もここから取得します |
| トピック用インデックス | `ft-published-topics-<ハッシュ>`（`TOPIC_INDEX`） | トピック ID と重心ベクトル。階層を表すグラフ構造の定義（`graphProfile`）も持ちます | ナレッジ ソースは直接指定しません。チャンク用インデックスのベクトル フィールドの `topicGraphProfile` から、`indexName` を経由して参照されます |

ナレッジ ソースとナレッジ ベースはインデックスではなく、検索対象と窓口を定義する設定です。

#### 各段階で起きること

| 段階 | 起きること | このノートブックでの指定 |
|---|---|---|
| ① 要求 | `retrieve` に質問と `topicGraph` を渡します。要求の `topicGraph` は、ナレッジ ソースに保存した設定をその要求だけ上書きします | `intents` に質問を 1 件。`minimal` は `messages` を受け付けません。`topicGraph.kind` を `none`・`balanced`・`diverse` で切り替えます |
| ② 計画 | 通常の `minimal` はモデルによる計画を省き、対象のナレッジ ソースをすべて検索します | 対象は専用ナレッジ ソース 1 つです。LLM は使いません |
| ③ ベクトル化 | 質問文を、検索フィールドのベクトライザーでベクトルに変換します | `snippet_vector` に設定したベクトライザーを使います |
| ④ 候補の取得 | チャンク用インデックスにベクトル検索を行い、近いチャンクを候補として集めます。`balanced` と `diverse` では、サンプリングの前に少なくとも `minDiversityPoolSize` 件を取得します | `minDiversityPoolSize = 1000`（指定できる範囲は 250〜10000）。`none` では指定しません |
| ⑤ トピック サンプリング | トピックの代表ベクトルを使い、複数のテーマから根拠を選びます。`balanced` は類似度と多様性を組み合わせる設定で、多様性係数は 0.2 の固定です。`diverse` は `diversityFactor`（0〜1）を要求ごとに指定します。`none` はこの段階を行いません | `diverse` の係数は 0.1 です |
| ⑥ 応答 | `extractiveData` では回答を生成せず、チャンクの抜粋（`response`）、出典（`references`）、実行履歴（`activity`）を返します | 参照、参照元データ、実行履歴をすべて有効にします |

#### 設定はどこから決まるか

| 設定 | 決まり方 |
|---|---|
| `topicGraph.kind` | 要求に `topicGraph` があればそれを使い、無ければナレッジ ソースの保存値を使います |
| `diversityFactor`（`diverse` のみ） | 要求で指定します。省略するとトピック プロファイルの値、それも無ければ 0.2 です。`balanced` は 0.2 の固定で、プロファイルの値とは無関係です |
| `minDiversityPoolSize`（`balanced`・`diverse`） | 要求で指定します。省略するとトピック プロファイルの値、それも無ければ 1000 です |

このノートブックは、ナレッジ ソースに `balanced`（候補数 1000）を保存し、2.5〜2.7 の各要求で `kind` と候補数を明示して上書きします。トピック プロファイルの `defaultKind` が、ナレッジ ベース経由の検索でどう使われるかは公式資料から確認できないため、プロファイルの既定値には頼りません。

出典: [Topic Graph の取得要求と設定値](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Retrieve API の `topicGraph` 定義](https://learn.microsoft.com/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-11-01-preview#balancedvectorquerytopicgraph) / [インデックス定義の `topicGraphProfile`](https://learn.microsoft.com/rest/api/searchservice/indexes/create-or-update?view=rest-searchservice-2026-11-01-preview) / [トピック グラフの概要（代表ベクトルと重心ベクトル）](https://learn.microsoft.com/azure/search/knowledge-graph-features-overview#compare-topic-graphs-and-entity-graphs) / [推論レベル minimal の動作](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort) / [取得手順と応答](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve) / [クエリ時のベクトル化](https://learn.microsoft.com/azure/search/vector-search-how-to-configure-vectorizer)

### 2 つのインデックスをつなぐもの

**2 つのインデックスをつなぐのは、設定上のインデックス名の参照です。チャンクの `uid` のような文書のキーを、結合キーにしているわけではありません。**

| 層 | つなぎ目 | 根拠 |
|---|---|---|
| 設定 | チャンク用インデックスのベクトル フィールドの `topicGraphProfile`（プロファイル名）が、同じインデックスの `vectorSearch.topicGraphProfiles` にある同名の項目を指し、その `indexName` がトピック用インデックスの名前です | 公式の REST 定義と前提条件に記載があります |
| データ | 公式資料は、トピック用インデックスがトピック ID と重心ベクトルを持つと説明しています。チャンクの `uid` とトピックを結ぶ結合キーの説明はありません | 公式資料。実際の中身は次のセルで確認します |

次のセルは読み取りだけで、2 点を確認します。保存済みのベクトルを使うため、埋め込みモデルは呼び出しません。

1. トピック用インデックスのうち、チャンクの ID を入れるフィールド（グラフ定義の `sourceChunkIdField` と `sourceParentIdField`）に値がある文書の数。
2. チャンク 1 件の保存済みベクトルで、トピック用インデックスのベクトル フィールドを検索した結果。

機能テスト 2 の既定の入力で作ったトピック用インデックスでは、チャンクの ID を持つ文書は 0 件でした。ベクトル フィールドには、述語 `HasCentroidVector` を持つトピック文書の重心ベクトルが入っており、チャンクのベクトルに近いトピックが返ります。つまり、このデータでは、チャンクとトピックの対応は ID ではなくベクトルの近さで表せます。他のデータでは件数や名前が変わります。

**`retrieve` がこの対応を内部でどう使うかは公開されていません。** このセルは、2 つのインデックスの中身がどう対応し得るかを確認するもので、サンプリングの手順を再現するものではありません。

出典: [インデックス定義の `topicGraphProfile`](https://learn.microsoft.com/rest/api/searchservice/indexes/create-or-update?view=rest-searchservice-2026-11-01-preview) / [Topic Graph の前提条件](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#prerequisites) / [トピック グラフの概要（トピック ID と重心ベクトル）](https://learn.microsoft.com/azure/search/knowledge-graph-features-overview#compare-topic-graphs-and-entity-graphs) / [ベクトル クエリ](https://learn.microsoft.com/azure/search/vector-search-how-to-query)

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    from azure.search.documents.models import VectorizedQuery

    graph_fields = topic_schema["graphProfile"]["fields"]
    topic_vector_field = next(field["name"] for field in topic_schema["fields"] if field.get("dimensions"))
    with SearchClient(SEARCH_ENDPOINT, CONTENT_INDEX, credential, **client_options) as content_client:
        probe = next(iter(content_client.search("*", select=["uid", CONTENT_FIELD, VECTOR_FIELD], top=1)))
    print("検索に使うチャンク:", probe["uid"], preview_text(" ".join(probe[CONTENT_FIELD].split())))
    with SearchClient(SEARCH_ENDPOINT, TOPIC_INDEX, credential, **client_options) as topic_client:
        id_counts = []
        for label, key in (("チャンク ID", "sourceChunkIdField"), ("親文書 ID", "sourceParentIdField")):
            found = topic_client.search("*", filter=f"{graph_fields[key]} ne null", top=0, include_total_count=True)
            id_counts.append({"確認する値": label, "フィールド名": graph_fields[key], "値を持つトピック文書数": found.get_count()})
        table(id_counts)
        nearest = topic_client.search(
            search_text=None, top=5,
            select=[graph_fields["subjectIdField"], graph_fields["subjectIdDisplayNameField"], graph_fields["predicateField"]],
            vector_queries=[VectorizedQuery(vector=probe[VECTOR_FIELD], k_nearest_neighbors=5, fields=topic_vector_field)],
        )
        table([{"類似度": round(hit["@search.score"], 4), "トピック ID": hit[graph_fields["subjectIdField"]],
                "トピック名": hit[graph_fields["subjectIdDisplayNameField"]], "述語": hit[graph_fields["predicateField"]]}
               for hit in nearest])
else:
    print("未実行: チャンク用インデックスとトピック用インデックスの対応確認。RUN_AZURE と ENABLE_TOPIC が必要です。")

### 2.4 同じ質問で選択方式だけを変える

| `topicGraph.kind` | 意味 | 指定できる値 |
|---|---|---|
| `none` | トピック サンプリングを無効化 | 多様性パラメーターは指定しません |
| `balanced` | 多様性係数 `0.2` の固定設定 | `minDiversityPoolSize` のみ |
| `diverse` | 多様性係数を要求単位で指定 | `diversityFactor`: 0〜1、`minDiversityPoolSize`: 250〜10000 |

プレビューでは、ナレッジ ベースが質問を見て自動的に `diverse` を選ぶわけではありません。要求で明示します。

**3 方式で変えるのは `topicGraph` だけです。** 質問、ナレッジ ソース、推論レベル `minimal`、出力形式 `extractiveData` は共通で、3 つの要求を同じ関数 `topic_request` で作ります。多様性の設定を同一条件で比較するため、推論レベルは `minimal` に固定します。`low`・`medium` にすると LLM による検索計画が入り、方式の違い以外の差が結果に混ざります。`minimal` には `messages` ではなく `intents` を渡します。以降の 3 セルは **それぞれ 1 回のライブ検索**を行います。

出典: [要求単位の Topic Graph 設定](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#override-topic-selection-for-one-request) / [公開されている設定範囲](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#review-settings-and-verification-limits) / [推論レベル](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort)

In [ ]:
TOPIC_QUESTION = "文書全体には、歴史、建造物、年中行事、観光についてどのような主要テーマがありますか。"
topic_results = {}


def topic_request(question: str, kind: str, diversity_factor: float = 0.1, minimum_pool: int = 1000) -> KnowledgeBaseRetrievalRequest:
    if not question.strip() or kind not in ("none", "balanced", "diverse"):
        raise ValueError("質問と topicGraph.kind を確認してください。")
    if kind != "none":
        if not 250 <= minimum_pool <= 10000:
            raise ValueError("minDiversityPoolSize は 250〜10000 です。")
    if kind == "diverse":
        if not 0 <= diversity_factor <= 1:
            raise ValueError("diversityFactor は 0〜1 です。")
    topic_parameters: NoneVectorQueryTopicGraph | BalancedVectorQueryTopicGraph | DiverseVectorQueryTopicGraph
    if kind == "none":
        topic_parameters = NoneVectorQueryTopicGraph()
    elif kind == "balanced":
        topic_parameters = BalancedVectorQueryTopicGraph(min_diversity_pool_size=minimum_pool)
    else:
        topic_parameters = DiverseVectorQueryTopicGraph(diversity_factor=diversity_factor, min_diversity_pool_size=minimum_pool)
    return KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=question)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(), output_mode="extractiveData", include_activity=True,
        knowledge_source_params=[SearchIndexKnowledgeSourceParams(
            knowledge_source_name=TOPIC_SOURCE, include_references=True, include_reference_source_data=True,
            topic_graph=topic_parameters)],
    )

### 2.4.1 検索結果を読みやすく表示する

`response[].content[].text` の中にある抽出結果の JSON を展開し、参照 ID で `references` と対応付けます。表示する原文は、検索応答が実際に返した断片です。参照側の長い全文で置き換えません。

2.5〜2.7 は **質問・検索設定 → 件数 → 文書ごとの抜粋と出典**の順に表示します。方式間で見比べるには 2.8 の比較表を使います。検索セルと表示セルは分けてあるため、取得後に表示セルだけ再実行しても再検索しません。抜粋は先頭 200 文字、表示件数は既定で 5 件です。追加の断片を見たい場合は `display_topic_result(topic_results["none"], "none", limit=10)` のように呼びます。

元の応答は `topic_results` に保持します。長いチャンク ID は折りたたんだ識別情報で確認できます。出典が URL ならリンク、ファイル名ならその名前を表示します。参照や出典が欠けている場合は「未記録」と表示し、内容から出典を推測しません。

出典: [抽出結果・activity・references の構成](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve?pivots=python#review-the-response)

In [ ]:
TOPIC_MODE_LABELS = {"none": "トピック選択なし", "balanced": "固定係数で選択", "diverse": "指定係数で選択"}


def topic_passages(result: dict) -> list[dict]:
    references = {str(reference["id"]): reference for reference in result.get("references", [])}
    passages = []
    for message in result["response"]:
        for part in message.get("content", []):
            if part.get("type") != "text":
                print("この表示では扱わないコンテンツ種類:", part.get("type"))
                continue
            text = part.get("text")
            if not isinstance(text, str):
                raise ValueError("検索応答の text が文字列ではありません。")
            items = json.loads(text) if text.lstrip().startswith(("[", "{")) else [{"content": text}]
            if isinstance(items, dict):
                items = [items]
            if not isinstance(items, list):
                raise ValueError("抽出結果の JSON は配列またはオブジェクトである必要があります。")
            for item in items:
                if not isinstance(item, dict) or not isinstance(item.get("content"), str):
                    raise ValueError("抽出結果に文字列の content がありません。")
                reference_id = item.get("ref_id")
                reference = references.get(str(reference_id), {})
                source_data = reference.get("sourceData")
                source_data = source_data if isinstance(source_data, dict) else {}
                body = item["content"]
                first_line, separator, remainder = body.partition("\n")
                title = item.get("title")
                if separator and first_line.strip().lower().endswith(".txt"):
                    title = title or first_line.strip()
                    body = remainder
                passages.append({"参照 ID": reference_id, "文書": title or "検索結果",
                                 "原文": body, "原文 URL": source_data.get("blob_url"),
                                 "uid": source_data.get("uid"), "activity ID": reference.get("activitySource")})
    return passages


def display_topic_request(request: KnowledgeBaseRetrievalRequest) -> None:
    payload = request.as_dict()
    parameters = payload["knowledgeSourceParams"][0]["topicGraph"]
    table([{"質問": payload["intents"][0]["search"], "方式": parameters["kind"],
            "トピック設定": parameters, "検索ソース": payload["knowledgeSourceParams"][0]["knowledgeSourceName"]}])


def topic_excerpt(body: str) -> str:
    return str(preview_text(" ".join(body.split())))


def topic_source_link(source: object) -> str:
    if source is None:
        return "出典未記録"
    escaped = html.escape(str(source))
    parsed_url = urlsplit(source) if isinstance(source, str) else None
    if parsed_url is not None and parsed_url.scheme in ("https", "http") and parsed_url.hostname:
        return f'<a href="{escaped}" target="_blank" rel="noopener noreferrer">出典を開く</a>'
    return f"出典: {escaped}"


def display_topic_result(result: dict, kind: str, limit: int = 5) -> None:
    if limit < 1:
        raise ValueError("表示件数は 1 以上にしてください。")
    passages = topic_passages(result)
    print(f"{TOPIC_MODE_LABELS[kind]} ({kind})")
    table([{"返却断片数": len(passages), "出典文書数（取得分）": len({row["原文 URL"] for row in passages if row["原文 URL"]}),
            "表示件数": min(limit, len(passages)), "出典未記録": sum(row["原文 URL"] is None for row in passages)}])
    if not passages:
        print("返却された本文は 0 件です。検索設定と実行履歴を確認してください。")
    for row in passages[:limit]:
        excerpt = topic_excerpt(row["原文"])
        reference_id = row["参照 ID"] if row["参照 ID"] is not None else "未記録"
        source_link = topic_source_link(row["原文 URL"])
        identifiers = {key: value if value is not None else "未記録" for key, value in {"チャンク ID": row["uid"], "activity ID": row["activity ID"]}.items()}
        display(HTML(f'<section style="border:1px solid #ccd5df;border-radius:8px;padding:12px;margin:8px 0">'
                     f'<strong>参照 {html.escape(str(reference_id))}: {html.escape(str(row["文書"]))}</strong>'
                     f'<p style="line-height:1.7">{html.escape(excerpt)}</p><p>{source_link}</p>'
                     f'<details><summary>参照の識別情報</summary><pre>{html.escape(json.dumps(identifiers, ensure_ascii=False, indent=2))}</pre></details></section>'))
    if len(passages) > limit:
        print(f"先頭 {limit} / {len(passages)} 件を表示しています。limit を増やすと残りの抜粋も表示できます。")


def display_topic_rank_table(passages_by_mode: dict[str, list[dict]], limit: int = 5) -> None:
    if limit < 1:
        raise ValueError("表示件数は 1 以上にしてください。")
    count = min(limit, max((len(rows) for rows in passages_by_mode.values()), default=0))
    if not count:
        print("比較表に表示する返却断片は 0 件です。")
        return
    header = '<th scope="col" style="width:4em">返却順</th>' + "".join(
        f'<th scope="col">{kind}</th>' for kind in TOPIC_MODE_LABELS)
    body = []
    for index in range(count):
        cells = [f'<th scope="row" style="vertical-align:top">{index + 1}</th>']
        for kind in TOPIC_MODE_LABELS:
            passages = passages_by_mode.get(kind)
            if passages is None:
                entry = "未実行"
            elif index >= len(passages):
                entry = "該当なし"
            else:
                row = passages[index]
                title = html.escape(str(row["文書"]))
                excerpt = html.escape(topic_excerpt(row["原文"]))
                entry = f'<strong>{title}</strong><br>{excerpt}<br>{topic_source_link(row["原文 URL"])}'
            cells.append(f'<td style="text-align:left;vertical-align:top;overflow-wrap:anywhere">{entry}</td>')
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f'<table style="width:100%;table-layout:fixed">'
                 f'<caption>返却順の比較（各方式の先頭 {limit} 件まで）</caption>'
                 f'<thead><tr>{header}</tr></thead><tbody>{"".join(body)}</tbody></table>'))

### 2.5 基準: トピック選択なし

`none` は Topic Graph を使わない比較用の要求です。

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    none_payload = topic_request(TOPIC_QUESTION, "none")
    display_topic_request(none_payload)
    topic_results["none"] = topic_kb_client.retrieve(none_payload, raw_response_hook=reject_partial_response).as_dict()
    print("none の結果を取得しました。次の表示セルで抜粋と出典を確認します。")
else:
    print("未実行: トピック選択なしのライブ検索。")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    if "none" not in topic_results:
        raise RuntimeError("先に 2.5 の検索セルを実行してください。")
    display_topic_result(topic_results["none"], "none")
else:
    print("未実行: none の結果表示。")

### 2.6 固定の多様性係数で検索する

`balanced` には `diversityFactor` を付けず、固定値 `0.2` を使います。

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    balanced_payload = topic_request(TOPIC_QUESTION, "balanced")
    display_topic_request(balanced_payload)
    topic_results["balanced"] = topic_kb_client.retrieve(balanced_payload, raw_response_hook=reject_partial_response).as_dict()
    print("balanced の結果を取得しました。次の表示セルで抜粋と出典を確認します。")
else:
    print("未実行: balanced のライブ検索。")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    if "balanced" not in topic_results:
        raise RuntimeError("先に 2.6 の検索セルを実行してください。")
    display_topic_result(topic_results["balanced"], "balanced")
else:
    print("未実行: balanced の結果表示。")

### 2.7 多様性係数を明示して検索する

公式例と同じ `diversityFactor = 0.1` を要求に設定します。この値が今回の文書群に最適だとは仮定しません。

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    diverse_payload = topic_request(TOPIC_QUESTION, "diverse", diversity_factor=0.1)
    display_topic_request(diverse_payload)
    topic_results["diverse"] = topic_kb_client.retrieve(diverse_payload, raw_response_hook=reject_partial_response).as_dict()
    print("diverse の結果を取得しました。次の表示セルで抜粋と出典を確認します。")
else:
    print("未実行: diverse のライブ検索。")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    if "diverse" not in topic_results:
        raise RuntimeError("先に 2.7 の検索セルを実行してください。")
    display_topic_result(topic_results["diverse"], "diverse")
else:
    print("未実行: diverse の結果表示。")

### 2.8 `kind` ごとの検索結果を並べて比較する

2.5〜2.7 で取得した応答から、返却断片数と出典文書数を集計し、`none`・`balanced`・`diverse` を列にして文書名・抜粋・出典を見比べます。既定で各方式の先頭 5 件を並べます。10 件に増やすには、下の呼び出しを `display_topic_rank_table(passages_by_mode, limit=10)` に変えます。同じ行は同じ返却位置を示すもので、同じ文書やチャンクを意味しません。

一部の方式だけを検索した場合も表示できます。「未実行」は応答がない方式、「該当なし」はその返却位置に断片がないことを示します。表示時に再検索しません。質問を変えた場合は、比較する方式を同じ質問で検索し直してください。

`none` を検索済みなら、共通するチャンク数と基準にない断片も確認できます。比較キーは検索ごとに変わり得る参照 ID ではなく `uid` です。未記録の `uid` は比較から除外し、その件数を表示します。チャンクの違いだけから、テーマの網羅性や正確性が改善したとは判断できません。

出典: [Topic Graph の確認範囲](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#review-settings-and-verification-limits) / [検索時の activity と references](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)

In [ ]:
if topic_results:
    passages_by_mode = {kind: topic_passages(topic_results[kind]) for kind in TOPIC_MODE_LABELS if kind in topic_results}
    uid_sets = {kind: {row["uid"] for row in passages if row["uid"] is not None}
                for kind, passages in passages_by_mode.items()}
    baseline = uid_sets.get("none")
    table([{"方式": kind, "返却断片数": len(passages_by_mode[kind]),
            "出典文書数（取得分）": len({row["原文 URL"] for row in passages_by_mode[kind] if row["原文 URL"]}),
            "確認済み固有チャンク数": len(uids), "none と共通": len(uids & baseline) if baseline is not None else "未集計",
            "none にないチャンク数": len(uids - baseline) if baseline is not None else "未集計",
            "uid 未記録": sum(row["uid"] is None for row in passages_by_mode[kind])}
           for kind, uids in uid_sets.items()])
    display_topic_rank_table(passages_by_mode)
    if baseline is None:
        print("基準 none の検索後にチャンクの差分を表示します。")
    elif len(passages_by_mode) == 1:
        print("balanced または diverse の検索後に基準 none との差分を表示します。")
    else:
        differences = [{"方式": kind, "参照 ID": row["参照 ID"], "文書": row["文書"], "抜粋": topic_excerpt(row["原文"])}
                       for kind, passages in passages_by_mode.items() if kind != "none" for row in passages
                       if row["uid"] is not None and row["uid"] not in baseline]
        if differences:
            print("基準 none にない断片の抜粋:")
            table(differences[:5])
            if len(differences) > 5:
                print(f"差分の先頭 5 / {len(differences)} 件を表示しています。全文は topic_results に残ります。")
        else:
            print("確認済みチャンク ID の範囲では、基準 none にない断片はありません。")
else:
    print("未実行: 比較する方式の検索セル（2.5〜2.7）を先に実行してください。")

### 2.8.1 各方式の実行履歴を確認する

取得済みの方式について、処理一覧と所要時間の図を表示します。`activity` の元 JSON は折りたたんで保持します。

このナレッジ ベースには Topic Graph を使う `searchIndex` 型のナレッジ ソースだけを接続しています。`activity[].type` に `searchIndex` が記録され、`graphSearch` が出ないのは正常です。`graphSearch` がないことは、Topic Graph が使われていないという意味ではありません。逆に、`searchIndex` の記録だけでは、どのトピックが候補選択に使われたかまでは確認できません。

Entity Graph 対応のナレッジ ソースで関係を検索するときは、ナレッジ ベースが質問を GQL に変換して実行します。また、トピック用インデックスに対する `docs/search.graph` での直接 GQL 実行も、このノートブックの `retrieve` とは別の操作です。

出典: [Topic Graph の検索構成](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Retrieve API の実行履歴](https://learn.microsoft.com/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-11-01-preview) / [Entity Graph をナレッジ ベース経由で検索する](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-through-a-knowledge-base) / [GQL を直接実行する API](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
import math
from datetime import datetime


def display_activity(activities: list[dict], title: str, *, show_raw: bool = True) -> None:
    if not isinstance(activities, list):
        raise TypeError("activity は配列である必要があります。SDK の応答を確認してください。")
    print(title, "activity 全件:")
    if show_raw:
        show_json(activities)
    else:
        raw = html.escape(json.dumps(activities, ensure_ascii=False, indent=2))
        display(HTML(f'<details><summary>activity の元 JSON を開く</summary><pre>{raw}</pre></details>'))
    if not activities:
        print("activity は 0 件です。処理内容を確認できません。include_activity を確認してください。")
        return
    labels = {
        "modelQueryPlanning": "検索計画の生成", "graphSearch": "GQL 生成・グラフ検索",
        "searchIndex": "インデックス検索", "azureBlob": "Blob ソース検索",
        "agenticReasoning": "推論設定・使用量", "modelAnswerSynthesis": "回答生成",
        "modelWebSummarization": "Web 検索結果の要約", "imageServing": "画像の取得・受け渡し",
    }

    def timestamp(value: str | None) -> datetime | None:
        if value is None:
            return None
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        if parsed.tzinfo is None:
            raise ValueError("activity の時刻にタイムゾーンがありません。")
        return parsed

    rows, records = [], []
    for activity in activities:
        kind = activity.get("type") or "未記録"
        graph = activity.get("graphSearchArguments") or {}
        model = activity.get("model") or {}
        start, end = timestamp(activity.get("startedAt")), timestamp(activity.get("completedAt"))
        if start is not None and end is not None and end < start:
            raise ValueError(f"activity {activity.get('id')}: 終了時刻が開始時刻より前です。")
        elapsed = activity.get("elapsedMs")
        basis = "elapsedMs"
        if elapsed is None and start is not None and end is not None:
            elapsed = (end - start).total_seconds() * 1000
            basis = "開始・終了時刻から算出"
        if elapsed is not None and (isinstance(elapsed, bool) or not isinstance(elapsed, (int, float)) or not math.isfinite(elapsed) or elapsed < 0):
            raise ValueError(f"activity {activity.get('id')}: elapsedMs が不正です。")

        def metric(key: str) -> object:
            value = activity.get(key)
            return graph.get(key) if value is None else value

        row = {
            "ID": activity.get("id"), "種類": kind, "処理": labels.get(kind, "未登録の処理種類"),
            "ソース": activity.get("knowledgeSourceName"), "モデル": model.get("modelName") or graph.get("modelName"),
            "時間 ms": elapsed, "時間の根拠": basis if elapsed is not None else None,
            "モデル時間 ms": graph.get("modelElapsedMs"),
            "入力トークン": metric("inputTokens"), "出力トークン": metric("outputTokens"),
            "推論トークン": activity.get("reasoningTokens"), "件数": activity.get("count"),
            "試行数": graph.get("attempts"), "成否・エラー": activity.get("error") or graph.get("outcome"),
        }
        rows.append({key: "未記録" if value is None else value for key, value in row.items()})
        records.append((activity, start, end, elapsed))
    table(rows)
    bounds = [(start, end) for _, start, end, _ in records if start is not None and end is not None]
    origin = min(start for start, _ in bounds) if bounds else None
    span = (max(end for _, end in bounds) - origin).total_seconds() * 1000 if origin is not None else max((elapsed for _, _, _, elapsed in records if elapsed is not None), default=0)
    caption = "タイムライン: 最初の開始時刻からの経過 ms" if origin is not None else "時間比較: 実際の開始時刻・順番・並列性は未記録"
    print(caption)
    left, plot_width, height = 330, 550, 95 + 44 * len(records)
    scale = plot_width / max(span, 1)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1040 {height}" width="100%" role="img" aria-label="{html.escape(title)}の処理時間">',
             f'<text x="10" y="22" font-family="sans-serif" font-size="14">{html.escape(caption)}</text>']
    ticks = [span * index / 4 for index in range(5)] if span else [0]
    for tick in ticks:
        x = left + tick * scale
        parts.append(f'<line x1="{x}" y1="45" x2="{x}" y2="{height - 25}" stroke="#ddd"/>')
        parts.append(f'<text x="{x}" y="40" text-anchor="middle" font-family="sans-serif" font-size="12">{tick:.1f}</text>')
    for index, (activity, start, end, elapsed) in enumerate(records):
        y = 70 + 44 * index
        kind = activity.get("type") or "未記録"
        label = f"{activity.get('id', '未記録')}: {labels.get(kind, kind)}"
        parts.append(f'<text x="10" y="{y + 5}" font-family="sans-serif" font-size="13">{html.escape(label)}</text>')
        if origin is not None and start is not None and end is not None:
            offset, duration = (start - origin).total_seconds() * 1000, (end - start).total_seconds() * 1000
        elif origin is None and elapsed is not None:
            offset, duration = 0, elapsed
        else:
            parts.append(f'<text x="{left}" y="{y + 5}" fill="#64748b" font-family="sans-serif" font-size="12">開始・終了時刻未記録</text>')
            continue
        x, width = left + offset * scale, duration * scale
        color = "#dc2626" if activity.get("error") or ((activity.get("graphSearchArguments") or {}).get("outcome") not in (None, "Success")) else "#3976b8"
        if duration == 0:
            parts.append(f'<circle cx="{x}" cy="{y}" r="4" fill="{color}"/>')
        else:
            parts.append(f'<rect x="{x}" y="{y - 9}" width="{width}" height="18" rx="3" fill="{color}"/>')
        parts.append(f'<text x="{left + plot_width + 12}" y="{y + 5}" font-family="sans-serif" font-size="12">{duration:.1f} ms</text>')
    display(SVG("".join(parts) + "</svg>"))

In [ ]:
if topic_results:
    for kind in TOPIC_MODE_LABELS:
        if kind in topic_results:
            display_activity(topic_results[kind]["activity"], f"Topic Graph: {kind}", show_raw=False)
else:
    print("未実行: 実行履歴を確認する方式の検索セル（2.5〜2.7）を先に実行してください。")

## 3. 結果の読み方と切り分け

| 観察した状態 | 次に確認すること |
|---|---|
| `403` | 実行ユーザーの Search 権限と、Search のマネージド ID のモデル権限、ネットワーク条件を分けて確認します |
| `206` | 部分応答です。この Notebook は成功として受け入れません。表示された本文と要求 ID から調べます |
| `412` | 計画後に設定が変更された可能性があります。2.1 から定義を読み直して新しい計画を確認します。無条件の PUT に変えません |
| 原文がない | `includeReferenceSourceData` とソースの原文フィールドを確認します |
| Topic Graph が接続されていない | 検索対象フィールドの `topicGraphProfile` → プロファイルの `indexName`、ベクトライザーを確認します |
| Topic の 3 方式が同じ | 少量の文書では差が出ないことがあります。差がないことを失敗とも、同じ品質の証明とも解釈しません |

### 保存される変更と終了後の扱い

ライブ実行で初回書き込みを承認した場合は、専用の `ft-topic-agentic-ks`・`ft-topic-agentic-kb` と、`ft-topic-src` の検索設定が残ります。トピック文書は再登録しません。設定変更はカーネルを終了しても残ります。

削除や設定の復元は自動化していません。不要になった場合は、この Notebook が新規作成した対象と依存関係を確認してから、専用 KB、専用ソースの順に片付けてください。`ft-topic-src` は機能テスト 3-B でも使うため削除しません。復元する場合も最新の定義と ETag を確認し、後から加わった変更を上書きしないでください。

Notebook の実行結果には原文と接続先が含まれ得ます。共有・コミット前に出力をクリアしてください。

出典: [Topic Graph の障害切り分け](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#troubleshooting) / [検索時の権限](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#prerequisites) / [同時実行制御](https://learn.microsoft.com/azure/search/search-howto-concurrency)

In [ ]:
if RUN_AZURE:
    for client in azure_clients:
        client.close()
    credential.close()
    print("SDK と資格情報のクライアントを閉じました。Azure リソースは削除していません。")
else:
    print("Azure の資格情報クライアントは作成していません。")